# GRPO Training â€” Qwen2.5-3B-Instruct

**Data Pipeline Incident Response Â· Meta PyTorch OpenEnv Hackathon**

Two-stage training: SFT warm-start â†’ GRPO reinforcement learning.

- Model: `Qwen/Qwen2.5-3B-Instruct` (text-only, 4-bit quantized)
- Hardware: Kaggle T4 (16GB VRAM)
- Expected runtime: ~15 min SFT + ~45 min GRPO


## 1. Setup

In [1]:
!pip install -qU --no-cache-dir unsloth unsloth_zoo transformers trl peft accelerate bitsandbytes \
    datasets pandas numpy openai python-dotenv
!rm -rf /kaggle/working/unsloth_compiled_cache
!rm -rf /kaggle/working/Meta_hackathon/unsloth_compiled_cache
print('Installation and Cache Clearance complete.')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.
Installation complete.


In [2]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = _s.get_secret('GITHUB_TOKEN_DEVARMANI')
    HF_TOKEN = _s.get_secret('HF_TOKEN')
except Exception:
    GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
    HF_TOKEN = os.getenv('HF_TOKEN', '')

REPO_DIR = '/kaggle/working/Meta_hackathon'

if not os.path.exists(REPO_DIR):
    !git clone -b dev/pratham https://{GITHUB_TOKEN}@github.com/standing-on-giants/Meta_hackathon.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout dev/pratham
    !git pull origin dev/pratham

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

Cloning into '/kaggle/working/Meta_hackathon'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 316 (delta 202), reused 251 (delta 141), pack-reused 0 (from 0)
Receiving objects: 100% (316/316), 510.24 KiB | 18.22 MiB/s, done.
Resolving deltas: 100% (202/202), done.
/kaggle/working/Meta_hackathon
Working directory: /kaggle/working/Meta_hackathon


In [3]:
import sys, json, textwrap, random, re, torch, numpy as np
from pathlib import Path
sys.path.insert(0, REPO_DIR)
from src.environment import DataPipelineEnv
from src.models import PipelineAction, PipelineObservation
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')


PyTorch 2.10.0+cu128, CUDA: True
GPU: Tesla T4, VRAM: 15.6GB


## 2. Load Qwen2.5-3B (4-bit quantization, LoRA r=32)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
    load_in_8bit=True,
    token=HF_TOKEN,
)
print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters()/1e9:.2f}B')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: Qwen/Qwen2.5-3B-Instruct
Parameters: 3.09B


In [5]:
# LoRA rank=32 (smaller model can afford higher rank)
model = FastLanguageModel.get_peft_model(
    model, r=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)')


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable: 59.9M / 3.15B (1.90%)


## 3. Stage 1 â€” SFT on Gold Trajectories

We train on hard-coded correct action sequences for easy/medium tasks.
This teaches the model output format and basic diagnostic strategy.


In [6]:
SYSTEM_PROMPT = textwrap.dedent('''
You are an expert data engineer diagnosing and fixing broken data pipelines.
You receive pipeline state and must choose ONE action per turn.
WORKFLOW: 1. read_data_sample 2. check_schema/compare_schema 3. Apply fix 4. run_pipeline
RULES: Respond with ONLY a JSON object. Never repeat failing actions. dedup for uniqueness failures.
''').strip()

GOLD_ACTIONS = {
    'easy': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_orders', 'n_rows': 20}},
        {'action_type': 'add_data_filter', 'params': {'step_id': 'transform_orders', 'filter_condition': 'user_id IS NOT NULL'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'medium': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_order_items', 'n_rows': 20}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_items', 'patch_type': 'dedup', 'column': 'order_item_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'hard': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_ads_insights', 'n_rows': 20}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_ads', 'patch_type': 'parse_currency', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_ads', 'patch_type': 'coalesce', 'column': 'spend'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'hard2': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_campaigns', 'n_rows': 20}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_campaigns'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'join_campaigns', 'patch_type': 'strip_prefix', 'column': 'campaign_id'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'join_campaigns', 'patch_type': 'cast_column', 'column': 'campaign_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
}

def format_obs(obs, step):
    failed = '\n'.join(f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}' for r in obs.failed_assertions) or '  (none)'
    passed = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag = '\n'.join(f'  {n.step_id}: {n.input_table} -> {n.output_table}' for n in obs.dag_structure)
    hist = '\n'.join(f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs)
    schema = ''
    if obs.current_schema: schema += '\nSCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff: schema += '\nDIFF: ' + json.dumps(obs.schema_diff)
    sample = ''
    if obs.data_sample: sample = '\nDATA: ' + json.dumps(obs.data_sample[:3], default=str)
    actions = '\n'.join(f'  {a}' for a in obs.actions_taken[-4:]) or '  (none)'
    return f'STEP {step}/{obs.max_steps} | TASK: {obs.task_id} ({obs.difficulty})\nDESCRIPTION: {obs.description}\nPIPELINE PASSED: {obs.pipeline_passed}\nLAST RESULT: {obs.last_action_result}\nDAG:\n{dag}\nFAILING:\n{failed}\nPASSING: {passed}\nHISTORY:\n{hist}\nACTIONS:\n{actions}{sample}{schema}\nRespond with ONE action JSON.'

def collect_gold(task_ids=['easy','medium', 'hard', 'hard2'], n_ep=10):
    pairs = []
    for tid in task_ids:
        gold = GOLD_ACTIONS.get(tid, [])
        if not gold: continue
        for _ in range(n_ep):
            env = DataPipelineEnv(task_id=tid)
            obs = env.reset()
            for si, ad in enumerate(gold, 1):
                pairs.append((format_obs(obs, si), json.dumps(ad)))
                result = env.step(PipelineAction(**ad))
                obs = result.observation
                if obs.pipeline_passed: break
    return pairs

gold_pairs = collect_gold(n_ep=10)
print(f'Collected {len(gold_pairs)} SFT pairs')
print(f'Sample action: {gold_pairs[0][1]}')


Collected 160 SFT pairs
Sample action: {"action_type": "read_data_sample", "params": {"table": "raw_orders", "n_rows": 20}}


In [7]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

sft_texts = [
    tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM_PROMPT},
         {'role':'user','content':obs},
         {'role':'assistant','content':act}],
        tokenize=False, add_generation_prompt=False)
    for obs, act in gold_pairs
]
sft_ds = Dataset.from_dict({'text': sft_texts})

SFT_DIR = '/kaggle/working/sft_qwen'
sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=sft_ds, dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
            average_tokens_across_devices=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=5,  # more epochs for smaller model
        warmup_ratio=0.1, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5, optim='adamw_8bit',
        weight_decay=0.01, lr_scheduler_type='cosine',
        output_dir=SFT_DIR, save_steps=50, seed=42,
    ),
)
print('Starting SFT...')
sft_stats = sft_trainer.train()
print(f'SFT done. Loss: {sft_stats.training_loss:.4f}')
model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/160 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting SFT...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 160 | Num Epochs = 5 | Total steps = 50
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 56.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 13.29 GiB is allocated by PyTorch, and 1.11 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 4. Stage 2 â€” GRPO Training

Uses environment rewards directly. G=8 completions per prompt.
KL penalty (0.1) anchors to SFT checkpoint to prevent degenerate collapse.


In [ ]:
def parse_action(text):
    # Strip <think> tags (Qwen sometimes produces them)
    text = re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()
    if '```' in text:
        text = '\n'.join(l for l in text.split('\n') if not l.strip().startswith('```')).strip()
    start = text.find('{')
    if start == -1: return None
    end = text.rfind('}') + 1
    if end <= start: return None
    try:
        data = json.loads(text[start:end])
        if 'action_type' in data: return PipelineAction(**data)
    except: pass
    return None

def pipeline_reward_fn(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c if isinstance(c, str) else c[0].get('content', '')
        action = parse_action(text)
        if action is None:
            rewards.append(-0.3); continue
        reward = 0.3  # format bonus
        try:
            env = DataPipelineEnv(task_id='hard')
            obs = env.reset()
            result = env.step(action)
            reward += result.reward or 0.0
            if action.action_type == 'compare_schema':
                if result.observation.schema_diff and len(result.observation.schema_diff) > 0:
                    reward += 0.3
        except: reward -= 0.2
        rewards.append(float(reward))
    return rewards

# Sanity check
test_r = pipeline_reward_fn(['{"action_type": "read_data_sample", "params": {"table": "raw_orders", "n_rows": 10}}'])
print(f'Reward sanity check: {test_r} (should be > 0)')


In [ ]:
from src.tasks import TASKS as _available
grpo_task_ids = [t for t in ['hard', 'hard2'] if t in _available]
print(f'GRPO tasks: {grpo_task_ids}')

grpo_prompts = []
for tid in grpo_task_ids:
    for _ in range(25):
        env = DataPipelineEnv(task_id=tid)
        obs = env.reset()
        chat = tokenizer.apply_chat_template(
            [{'role':'system','content':SYSTEM_PROMPT},
             {'role':'user','content':format_obs(obs, 1)}],
            tokenize=False, add_generation_prompt=True)
        grpo_prompts.append({'prompt': chat})

grpo_ds = Dataset.from_list(grpo_prompts)
print(f'GRPO dataset: {len(grpo_ds)} prompts')


In [ ]:
from trl import GRPOConfig, GRPOTrainer

GRPO_DIR = '/kaggle/working/grpo_qwen'
grpo_config = GRPOConfig(
    report_to="none",
    average_tokens_across_devices=False,
    output_dir=GRPO_DIR,
    num_generations=4,       # Reduced from 8 to 4 to save VRAM
    max_completion_length=200,
    temperature=0.8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    beta=0.2, # Strong anchor to SFT
    loss_type='grpo',
    logging_steps=5, save_steps=50, seed=42,
    max_prompt_length=MAX_SEQ_LENGTH,
    max_grad_norm=1.0, warmup_ratio=0.05,
)

grpo_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    reward_funcs=pipeline_reward_fn,
    args=grpo_config, train_dataset=grpo_ds,
)

print(f'GRPO: KL={grpo_config.beta}, G={grpo_config.num_generations}')
print('Starting GRPO training...')
grpo_stats = grpo_trainer.train()
print(f'GRPO done. Loss: {grpo_stats.training_loss:.4f}')
model.save_pretrained(GRPO_DIR)
tokenizer.save_pretrained(GRPO_DIR)


## 5. Evaluation

In [ ]:
from unsloth import FastLanguageModel as FLM
FLM.for_inference(model)

def run_eval(model, tokenizer, task_id, max_steps=20):
    env = DataPipelineEnv(task_id=task_id)
    obs = env.reset(); step = 0
    for step in range(1, max_steps+1):
        if obs.pipeline_passed: break
        msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':format_obs(obs,step)}]
        inp = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to(model.device)
        with torch.no_grad():
            out = model.generate(inp, max_new_tokens=200, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        resp = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
        action = parse_action(resp)
        if action is None: action = PipelineAction(action_type='compare_schema', params={'table':'raw_orders'})
        result = env.step(action); obs = result.observation
        if result.done: break
    nt = len(obs.failed_assertions) + len(obs.passed_assertions)
    np_ = len(obs.passed_assertions)
    return {'task_id': task_id, 'score': round(min(max(np_/nt if nt>0 else 0, 0.01), 0.99), 3), 'passed': obs.pipeline_passed, 'steps': step}

eval_tasks = [t for t in ['easy','medium','hard','hard2'] if t in _available]
baseline = {'easy': 0.70, 'medium': 0.55, 'hard': 0.30, 'hard2': 0.30}

print(f"{'Task':<10}{'Base':<10}{'Trained':<10}{'Delta':<10}{'Pass?'}")
print('='*50)
for tid in eval_tasks:
    r = run_eval(model, tokenizer, tid)
    b = baseline.get(tid, 0)
    d = r['score'] - b
    s = '+' if d >= 0 else ''
    print(f"{tid:<10}{b:<10.3f}{r['score']:<10.3f}{s}{d:<9.3f}{'[PASSED]' if r['passed'] else '[FAILED]'}")


## 6. Push to Hub (optional)

In [ ]:
LOCAL_MERGED_DIR = '/kaggle/working/qwen-merged-16bit'
print(f'Saving merged 16-bit model locally to {LOCAL_MERGED_DIR}...')
model.save_pretrained_merged(LOCAL_MERGED_DIR, tokenizer, save_method='merged_16bit')


from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
GITHUB_TOKEN = _s.get_secret('GITHUB_TOKEN')
HF_TOKEN = _s.get_secret('hugging_face_access_token')

HF_REPO = 'Abhinav-hf/data-pipeline-incident-qwen-grpo'
if HF_TOKEN:
    print(f'Pushing to {HF_REPO}...')
    model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
    print(f'Done: https://huggingface.co/{HF_REPO}')
else:
    print('No HF_TOKEN — saved locally only.')
